In [2]:
# python 3.13.9

In [3]:
import numpy as np 
import pandas as pd
import sys
import os

In [4]:
data='/Users/Tysia/Desktop/Edukacja/Informatyka/Magisterka/ml-1m'
rating_data=os.path.join(data,'ratings.dat')
movie_data=os.path.join(data,'movies.dat')
user_data=os.path.join(data,'users.dat')

ratings=pd.read_csv(
  rating_data,
  sep='::',
  engine='python',
  names=['user','item','rating','timestamp'],
  encoding='latin-1'   
)

movies=pd.read_csv(
    movie_data, 
    sep='::',
    engine='python',
    names=['item', 'title', 'genres'],
    encoding='latin-1'
)
users = pd.read_csv(
    user_data, 
    sep='::',
    engine='python',
    names=['user', 'gender', 'age', 'occupation', 'zipcode'],
    encoding='latin-1'
)

In [4]:
print(ratings.head())
print(movies.head())
print(users.head())

   user  item  rating  timestamp
0     1  1193       5  978300760
1     1   661       3  978302109
2     1   914       3  978301968
3     1  3408       4  978300275
4     1  2355       5  978824291
   item                               title                        genres
0     1                    Toy Story (1995)   Animation|Children's|Comedy
1     2                      Jumanji (1995)  Adventure|Children's|Fantasy
2     3             Grumpier Old Men (1995)                Comedy|Romance
3     4            Waiting to Exhale (1995)                  Comedy|Drama
4     5  Father of the Bride Part II (1995)                        Comedy
   user gender  age  occupation zipcode
0     1      F    1          10   48067
1     2      M   56          16   70072
2     3      M   25          15   55117
3     4      M   45           7   02460
4     5      M   25          20   55455


In [5]:
path = '/Users/Tysia/Desktop/Edukacja/Informatyka/Magisterka/Daisyrec/libraries/DaisyRec-v2.0-dev'

sys.path.append(path)

In [6]:
import torch
import torch.utils.data as data
from sklearn.model_selection import train_test_split
from types import SimpleNamespace
import scipy.sparse as sp
from daisy.model.MFRecommender import MF
#from daisy.utils.loader import Sampler, PointSampler # Klasy do ładowania danych do modelu
from daisy.model.NeuMFRecommender import NeuMF
from daisy.utils.sampler import BasicNegtiveSampler
from daisy.utils.dataset import BasicDataset, CandidatesDataset, get_dataloader
from daisy.utils.utils import get_ur, build_candidates_set
from daisy.utils.metrics import Recall, NDCG, Precision, HR, MAP, MRR, calc_ranking_results
from tqdm import tqdm

c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# mapowanie ID - DaisyRec wymaga ID od 0 do N-1. Robimy to ręcznie, aby mieć kontrolę.
print("Mapowanie ID użytkowników i filmów...")
user_map = {u: i for i, u in enumerate(ratings['user'].unique())}
item_map = {i: j for j, i in enumerate(ratings['item'].unique())}

ratings['user'] = ratings['user'].map(user_map)
ratings['item'] = ratings['item'].map(item_map)

Mapowanie ID użytkowników i filmów...


In [8]:
import logging
# konfiguracja loggera, żeby biblioteka miała gdzie wypisywać informacje
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

In [10]:
config = {
    'algo_name': 'mf',
    
    # --- PARAMETRY Z MYMEDIALITE ---
    'factors': 10,        # MML: num_factors=10
    'epochs': 30,         # MML: num_iter=30
    'lr': 0.01,           # MML: learn_rate=0.01
    
    # DaisyRec rozdziela regularyzację na embeddingi (reg_1) i biasy (reg_2).
    # MyMediaLite używa jednej wartości dla wszystkiego, więc ustawiamy obie tak samo.
    'reg_1': 0.015,       # MML: regularization=0.015
    'reg_2': 0.015,       # MML: regularization=0.015
    
    # --- KLUCZOWE DLA ODZWOROWANIA MYMEDIALITE ---
    'loss_type': 'MSE',   # Ważne! MyMediaLite minimalizuje błąd kwadratowy (RMSE).
                          # Domyślnie DaisyRec może mieć 'BPR' lub 'CL' (ranking), 
                          # co dałoby zupełnie inne wyniki.
    
    'optimizer': 'sgd',   # MML używa SGD (Stochastic Gradient Descent)
    
    # Parametry techniczne
    'num_ng': 0,          # Dla MSE/Rating Prediction nie potrzebujemy negative sampling
    'batch_size': 256,    # Standardowa wartość (MML nie podaje, dobierz domyślną)
    'gpu': '0'
}

In [19]:
# mf
config = {
    # Podstawowe
    'algo_name': 'mf',
    'dataset': 'ml-1m',
    'user_num': len(user_map),
    'item_num': len(item_map),
    'UID_NAME': 'user',    # Nazwa kolumny usera
    'IID_NAME': 'item',    # Nazwa kolumny itemu
    'INTER_NAME': 'rating', # Nazwa kolumny oceny
    
    # Parametry treningu
    'factors': 10,         # Wymiar embeddingów
    'epochs': 30,          # Liczba epok
    'lr': 0.01,            # Learning rate
    'batch_size': 256,     # Rozmiar batcha
    'optimizer': 'adam',
    'loss_type': 'BPR',    # Pair-wise loss (najlepszy do rekomendacji)
    'reg_1': 0.015,          # L1 Regularization
    'reg_2': 0.015,        # L2 Regularization
    'dropout': 0.1,
    
    # Parametry próbkowania (Sampling)
    'sample_method': 'uniform', # Wymagane przez BasicNegtiveSampler
    'num_ng': 4,                # 4 negatywne przykłady na 1 pozytywny
    'sample_ratio': 0.0,
    
    # Ewaluacja
    'topk': 3675,                 # Długość listy rekomendacji
    'cand_num': len(item_map), # <-- FIX: Oceniamy wszystkie możliwe filmy
    'test_method': 'rsbr',     # <-- FIX: Potrzebne do nazewnictwa plików wyników
    'prepro': 'origin',        # <-- FIX: Potrzebne do nazewnictwa plików wyników
    'res_path': './results/',  # <-- FIX: Ścieżka zapisu wyników
    'metrics': ['recall', 'ndcg', 'precision', 'mrr'],
    # Inne
    'gpu': '0',                 # ID karty GPU
    'early_stop': True,
    'init_method': 'normal', 
    'tune_testset': False,
    'logger': logger
}

In [39]:
# NeuMF
config = {
    # Dane
    'algo_name': 'neumf',     
    'dataset': 'ml-1m',
    'user_num': len(user_map),
    'item_num': len(item_map),
    'UID_NAME': 'user',    
    'IID_NAME': 'item',    
    'INTER_NAME': 'rating', 
    
    # Parametry Modelu NeuMF
    'model_name': 'NeuMF-end', # Trenowanie End-to-End (od zera)
    'GMF_model': None,         # Brak pre-trenowanego GMF
    'MLP_model': None,         # Brak pre-trenowanego MLP
    'factors': 32,             # Wymiar embeddingów (GMF i MLP)
    'num_layers': 4,           # <--- NOWE: Liczba warstw MLP (np. 32 -> 16 -> 8)
    'dropout': 0.1,            # <--- NOWE: Dropout rate (0.1 to bezpieczna wartość)
    
    # Parametry Treningu
    'epochs': 20,              # NeuMF często potrzebuje więcej epok niż MF
    'lr': 0.001,               # Mniejszy learning rate jest lepszy dla sieci neuronowych
    'batch_size': 256,        # Większy batch size przyspiesza trening na GPU
    'optimizer': 'adam',
    'loss_type': 'BPR',        # BPR jest dobre do rankingu
    'reg_1': 0.0,
    'reg_2': 0.0001,           # L2 Regularization (zmniejszone dla sieci)
    'init_method': 'normal',
    'tune_testset': False,
    'early_stop': True,
    
    # Sampler
    'sample_method': 'uniform', 
    'sample_ratio': 0.0,
    'num_ng': 4,
    
    # Ewaluacja
    'topk': 5,
    'cand_num': len(item_map), 
    'test_method': 'rsbr',     
    'prepro': 'origin',       
    'res_path': './results/', 
    'metrics': ['recall', 'ndcg', 'precision', 'mrr'],
    
    # Systemowe
    'gpu': '0',
    'logger': logger  
}

In [20]:
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42, shuffle=True)

In [21]:
# Tworzymy słowniki interakcji (Ground Truth) potrzebne do metryk
print("Budowanie struktur Ground Truth...")
train_ur = get_ur(train_df) # User -> Items w treningu
test_ur = get_ur(test_df)   # User -> Items w teście
config['train_ur'] = train_ur # Model MF w DaisyRec czasem tego potrzebuje

Budowanie struktur Ground Truth...


In [22]:
# Przygotowanie danych treningowych
# Sampler generuje negatywne przykłady (u, pos_i, neg_i)
sampler = BasicNegtiveSampler(train_df, config)
train_samples = sampler.sampling()

# Dataset i DataLoader z DaisyRec
train_dataset = BasicDataset(train_samples)
train_loader = get_dataloader(train_dataset, batch_size=config['batch_size'], shuffle=True, num_workers=0)

c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\daisy\utils\sampler.py:91: FutureWarning: using <function BasicNegtiveSampler.sampling.<locals>.<lambda> at 0x000001ED5000B100> in Series.agg cannot aggregate and has been deprecated. Use Series.transform to keep behavior unchanged.
  self.df['neg_set'] = self.df[self.uid_name].agg(lambda u: js[u])


In [23]:
#mf
model = MF(config)
model.fit(train_loader)

[Epoch 030]: 100%|██████████| 12503/12503 [01:19<00:00, 156.74it/s]


In [36]:
# NeuMF
model = NeuMF(config)
model.fit(train_loader)

[Epoch 020]: 100%|██████████| 12503/12503 [14:55<00:00, 13.96it/s] 


In [24]:
# TESTOWANIE 
# dla każdego usera z testu przygotowuje listę filmów do oceny (usuwając te, które widział w treningu)
test_u, test_ucands = build_candidates_set(test_ur, train_ur, config)
test_dataset = CandidatesDataset(test_ucands)
test_loader = get_dataloader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

In [25]:
np.asfarray = np.asarray

In [26]:
#  mf
# calc_ranking_results to funkcja pomocnicza, która liczy wszystko naraz
predictions = model.rank(test_loader) 
results = calc_ranking_results(test_ur, predictions, test_u, config)
print(results)

INFO:root:Recall@10: 0.0003
INFO:root:NDCG@10: 0.0049
INFO:root:Precision@10: 0.0011
INFO:root:MRR@10: 0.0033


       KPI@K         1         5        10        20        30        50  \
0     Recall  0.000025  0.000081  0.000259  0.000899  0.001552  0.003146   
1       NDCG  0.001656  0.003240  0.004913  0.008524  0.012734  0.021675   
2  Precision  0.001656  0.001060  0.001060  0.001284  0.001733  0.002322   
3        MRR  0.001656  0.002666  0.003333  0.004288  0.005077  0.006229   

       3675  
0  0.999340  
1  0.288880  
2  0.009007  
3  0.010793  


In [ ]:
#MF
import torch
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. Ustawienia urządzenia (GPU/CPU)
# Sprawdzamy, czy w configu jest ustawione GPU i czy jest dostępne
device = torch.device('cuda' if torch.cuda.is_available() and config['gpu'] != '0' else 'cpu')
model = model.to(device)

# 2. Przełączamy model w tryb ewaluacji
model.eval()

# Listy na wyniki
y_true = []
y_pred = []

print("Obliczanie RMSE i MAE...")

# 3. Iteracja przez zbiór testowy
# Zakładam, że masz DataFrame 'test' (z kolumnami user, item, rating)
# Jeśli masz 'test_loader' (standardowy, nie Candidate), możesz iterować przez niego
# Ale najbezpieczniej (i najprościej w DaisyRec) jest wziąć dane wprost z DataFrame:

with torch.no_grad(): # Wyłączamy gradienty dla przyspieszenia
    # Konwertujemy dane testowe na tensory (cały batch naraz dla szybkości)
    # Upewnij się, że typy danych są poprawne (Long dla ID, Float dla Ocen)
    u_tensor = torch.tensor(test_df['user'].values, dtype=torch.long).to(device)
    i_tensor = torch.tensor(test_df['item'].values, dtype=torch.long).to(device)
    r_tensor = torch.tensor(test_df['rating'].values, dtype=torch.float).to(device)

    # Predykcja modelu
    # Model MF w DaisyRec w metodzie forward() przyjmuje (user_ids, item_ids)
    prediction_tensor = model(u_tensor, i_tensor)
    
    # Pobranie wyników do CPU
    y_pred = prediction_tensor.cpu().numpy().flatten()
    y_true = r_tensor.cpu().numpy().flatten()

# 4. Obliczenie metryk
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)

print("="*30)
print(f"WYNIKI DaisyRec (MF)")
print("="*30)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

Obliczanie RMSE i MAE...
WYNIKI DaisyRec (MF)
RMSE: 3.9562
MAE:  3.6329


In [40]:
# NeuMF calc_ranking_results to funkcja pomocnicza, która liczy wszystko naraz
predictions = model.rank(test_loader) 
results = calc_ranking_results(test_ur, predictions, test_u, config)
print(results)

       KPI@K         1         5
0     Recall  0.002428  0.008155
1       NDCG  0.052335  0.102739
2  Precision  0.052335  0.036005
3        MRR  0.052335  0.085911
